# Missing Data and Masking

This notebook explains linopy's NaN convention under v1 and how to handle missing data.

1. [The NaN convention](#the-nan-convention) — design principles
2. [What raises](#what-raises) — NaN at API boundaries
3. [Handling NaN with `.fillna()`](#handling-nan-with-fillna) — choosing the right fill value
4. [Masking constraints](#masking-constraints) — `.sel()` and `mask=`
5. [Masking with NaN in coefficients](#masking-with-nan-in-coefficients) — multi-dimensional patterns
6. [Legacy NaN behavior](#legacy-nan-behavior-for-comparison) — how it worked before

For coordinate alignment rules, see [Arithmetic Convention](arithmetic-convention.ipynb).

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import linopy

linopy.options["arithmetic_convention"] = "v1"

In [ ]:
m = linopy.Model()
time = pd.RangeIndex(5, name="time")
x = m.add_variables(lower=0, coords=[time], name="x")

# Data with NaN
data = xr.DataArray([1.0, np.nan, 3.0, 4.0, 5.0], dims=["time"], coords={"time": time})

---

## The NaN convention

In linopy v1, **NaN means "absent term."** It is never a numeric value.

### How NaN enters

Only two sources produce NaN inside linopy data structures:

1. **`mask=` argument** at construction (`add_variables`, `add_constraints`) — you explicitly declare which slots exist.
2. **Structural operations** that produce absent slots: `.shift()`, `.where()`, `.reindex()`, `.reindex_like()`, `.unstack()` (with missing combinations).

Operations that do **not** produce NaN: `.roll()` (circular), `.sel()` / `.isel()` (subset), `.drop_sel()` (drops), `.expand_dims()` / `.broadcast_like()` (broadcast existing data).

### How NaN propagates

An expression is a sum of terms. Each term has a coefficient, a variable reference, and the expression has a shared constant. NaN marks an **individual term** as absent — it does not mask the entire coordinate.

When expressions are combined (e.g., `x*2 + y.shift(time=1)`), each term is kept independently. At time=0, `y.shift` contributes no term (NaN coeffs, vars=-1), but `x*2` is still valid. The result at time=0 is `2*x[0]` — not absent.

A coordinate is only fully absent when **all** terms have vars=-1 **and** the constant is NaN. This is exactly what `isnull()` checks.

### Where NaN lives

NaN is burned directly into the float fields: `coeffs`, `const`, `rhs`, `lower`, `upper`. Integer fields (`labels`, `vars`) use **-1** as their equivalent sentinel. There is no separate boolean mask array.

### What raises

Any **user-supplied NaN at an API boundary** — in bounds, constants, factors, or RHS — raises `ValueError` immediately. Masking is always explicit via `mask=` or `.sel()`, never by passing NaN as a value.

### Why this is consistent

- **`lhs >= rhs` is `lhs - rhs >= 0`**, so RHS obeys the same rule as any constant — no special case.
- **No dual role for NaN**: it cannot mean both "absent" and "a number I computed with." Internal NaN (from `shift`, `mask=`) is always structural. User NaN is always an error.
- **Absent terms, not absent coordinates**: combining a valid expression with a partially-absent one does not destroy the valid part. Only when *every* term at a coordinate is absent is the coordinate itself absent.

---

## What raises

**NaN in any arithmetic operand raises `ValueError`.** This includes:
- Constants added/subtracted: `expr + data_with_nan`
- Factors multiplied/divided: `expr * data_with_nan`
- Constraint RHS: `expr >= data_with_nan` (because `expr >= rhs` is `expr - rhs >= 0`)

There is no implicit fill. The library does not guess whether NaN means "zero," "exclude," or "identity." You decide.

In [ ]:
# All of these raise ValueError:
for op_name, op_fn in [
    ("add", lambda: x + data),
    ("mul", lambda: x * data),
    ("constraint", lambda: x >= data),
]:
    try:
        op_fn()
    except ValueError:
        print(f"{op_name}: ValueError raised (NaN in operand)")

---

## Handling NaN with `.fillna()`

When your data contains NaN, fill it explicitly before combining with expressions. The fill value depends on what the NaN means in your context:

| Operation | Fill value | Meaning |
|-----------|-----------|--------|
| `expr + data.fillna(0)` | 0 | NaN = "no offset" |
| `expr * data.fillna(0)` | 0 | NaN = "exclude this term" |
| `expr * data.fillna(1)` | 1 | NaN = "no scaling" |
| `expr / data.fillna(1)` | 1 | NaN = "no scaling" |

The choice is yours — and that's the point. Under legacy, the library chose for you (0 for add/mul, 1 for div). Under v1, you make the decision explicit.

In [ ]:
# Fill NaN before operating — you choose the fill value
print("add fillna(0):", (x + data.fillna(0)).const.values)
print("mul fillna(0):", (x * data.fillna(0)).coeffs.squeeze().values)
print("mul fillna(1):", (x * data.fillna(1)).coeffs.squeeze().values)

---

## Masking constraints

A common pattern: your data has NaN at positions where no constraint should exist. For example, availability data that's only defined for certain hours, or cost data with missing entries.

### Approach 1: `.sel()` (preferred)

Select only valid positions — the constraint has fewer coordinates:

In [ ]:
# Availability data with NaN = "no limit at this hour"
availability = xr.DataArray(
    [100.0, 80.0, np.nan, np.nan, 60.0], dims=["time"], coords={"time": time}
)

# Select only where data is valid — constraint has fewer coordinates
valid = availability.notnull()
m.add_constraints(x.sel(time=valid) <= availability.sel(time=valid), name="avail")

No fillna, no mask parameter — the constraint simply doesn't exist at the NaN positions.

### Approach 2: `mask=` parameter

When `.sel()` is inconvenient (e.g., multi-dimensional data where NaN positions vary across dimensions), use `mask=`:

In [ ]:
# Same result using mask= instead of .sel()
mask = availability.notnull()
m.add_constraints(x <= availability.fillna(0), name="avail_masked", mask=mask)

The same approaches work for variables with NaN bounds:

```python
# With .sel()
valid = upper_bounds.notnull()
m.add_variables(upper=upper_bounds.sel(i=valid), coords=[valid_coords], name="y")

# Or with mask=
mask = upper_bounds.notnull()
m.add_variables(upper=upper_bounds.fillna(0), mask=mask, name="y")
```

---

## Masking with NaN in coefficients

When NaN appears in coefficient data (e.g., efficiency factors where some combinations don't apply), the same two approaches work:

In [ ]:
# Efficiency data: solar has no efficiency at night (NaN)
techs = pd.Index(["solar", "wind"], name="tech")
hours = pd.RangeIndex(4, name="hour")
gen = m.add_variables(lower=0, coords=[hours, techs], name="gen")

efficiency = xr.DataArray(
    [[np.nan, 0.35], [0.8, 0.35], [0.9, 0.35], [np.nan, 0.35]],
    dims=["hour", "tech"],
    coords={"hour": hours, "tech": techs},
)

# Approach 1: .sel() — select only valid hours per tech
valid_hours = efficiency.sel(tech="solar").notnull()
solar_gen = gen.sel(tech="solar", hour=valid_hours)
solar_eff = efficiency.sel(tech="solar", hour=valid_hours)
print("sel approach — solar hours:", solar_gen.coords["hour"].values)

# Approach 2: mask= — keep all coordinates, mask invalid ones
rhs = xr.DataArray([50.0] * 4, dims=["hour"], coords={"hour": hours})
coeff_mask = efficiency.notnull().all("tech")
expr = gen * efficiency.fillna(0)
m.add_constraints(expr >= rhs, name="min_output", mask=coeff_mask)
print("mask approach — constraint mask:", coeff_mask.values)

---

## Legacy NaN behavior (for comparison)

Under legacy, NaN was handled implicitly:
- **In arithmetic**: silently replaced with neutral elements (0 for add/sub/mul, 1 for div)
- **In constraint RHS**: NaN meant "no constraint here" — auto-masked internally
- **With `auto_mask=True`**: NaN in variable bounds meant "no variable here"

This was convenient but could mask data quality issues. A NaN from a data pipeline bug would silently become 0, producing a valid but wrong model.

### Migration

| Legacy code (silent) | v1 equivalent (explicit) |
|---|---|
| `x + data_with_nans` | `x + data_with_nans.fillna(0)` |
| `x * data_with_nans` | `x * data_with_nans.fillna(0)` |
| `x / data_with_nans` | `x / data_with_nans.fillna(1)` |
| `m.add_constraints(expr >= nan_rhs)` | `m.add_constraints(expr.sel(...) >= rhs.sel(...))` |
| `Model(auto_mask=True)` | Explicit `mask=` or `.sel()` |

---

## Summary

| Aspect | v1 | Legacy |
|---|---|---|
| **NaN means** | Absent term (not absent coordinate) | Numeric placeholder (filled silently) |
| **NaN sources** | `mask=`, structural ops only | Anywhere (user data, bounds, RHS) |
| **NaN in operands** | `ValueError` | Filled with neutral element (0 or 1) |
| **NaN in constraint RHS** | `ValueError` | Auto-masked |
| **Combining expressions** | Absent terms ignored, valid terms kept | NaN filled before combining |
| **Coordinate absent when** | All terms absent AND const is NaN | Never (NaN always filled) |
| **Masking** | Explicit: `.sel()` or `mask=` | Implicit via NaN / `auto_mask` |
| **Storage** | Float fields + `-1` sentinels | Same, but NaN has dual role |
| **Fill value choice** | User decides | Library decides |